# LTEM Contrast Proxy from 3D Magnetization Data

This notebook reads a `.vtk` file containing `(x, y, z)` voxel positions and a
magnetization vector `(Mx, My, Mz)` at each voxel, averages the magnetization
**through the sample thickness** (the z-axis), and plots the resulting
in-plane components `(Mx, My)` as a proxy for Lorentz TEM (LTEM) contrast.

**What this is / isn't:**
- This is the *simple proxy* version: a color-wheel + quiver map of the
  thickness-projected in-plane magnetization. This is the same quantity
  (the projected in-plane induction) that underlies real LTEM image
  formation, and is commonly used as a quick way to visualize a
  magnetization texture the way an LTEM experiment would "see" it
  (in-plane component only, integrated through thickness).
- It does **not** simulate the actual electron optics (phase shift,
  defocus, Fresnel fringes). If you later want a physics-based
  Mansuripur/TIE-style phase and defocused-image simulation, the
  `average_through_thickness` output here (`Mx_avg`, `My_avg` on a
  regular grid) is exactly the input that calculation needs, so this
  notebook is a drop-in first stage for that too.

**Grid handling:** the reader and averaging function work for both
structured (regular voxel grid) and unstructured/point-cloud VTK data —
no assumption is made about grid structure other than that each sample
has an (x, y, z) position and a 3-component vector.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pyvista as pv

## 1. Reader: load a magnetization vector field from a `.vtk` file

`read_vtk_magnetization` uses `pyvista` (which wraps VTK) to open the file,
so it transparently handles legacy `.vtk` as well as modern XML formats
(`.vts`, `.vtu`, `.vti`, ...) and both structured and unstructured grids.

It looks for the magnetization data in this order:
1. An explicit `vector_field_name` you pass in, if given.
2. A 3-component array under a common name (`M`, `Magnetization`, `moment`, `spin`, ...).
3. Any 3-component array in the file's point/cell data.
4. Three separate scalar arrays named like `Mx`/`My`/`Mz` (also tries `mx/my/mz`, `M_x/M_y/M_z`).

If none of that matches your file's naming, pass `vector_field_name=...`
explicitly, or check `mesh.point_data.keys()` / `mesh.cell_data.keys()`
(printed in the error message) to see what's actually in the file.

In [ ]:
def read_vtk_magnetization(filename, vector_field_name=None, prefer="point"):
    """
    Read (x,y,z) positions and a 3-component magnetization vector from a VTK file.

    Parameters
    ----------
    filename : str
        Path to a .vtk/.vts/.vtu/.vti/... file readable by pyvista.
    vector_field_name : str, optional
        Name of the vector array to use. If None, auto-detected.
    prefer : {"point", "cell"}
        Whether to prefer point data or cell data when both are present.

    Returns
    -------
    points : (N, 3) ndarray
    M : (N, 3) ndarray
    mesh : the pyvista mesh object (kept around in case you need it, e.g.
           mesh.bounds, mesh.dimensions, other fields, etc.)
    """
    mesh = pv.read(filename)

    point_data = mesh.point_data
    cell_data = mesh.cell_data

    if prefer == "point" and len(point_data.keys()) > 0:
        data, points = point_data, mesh.points
    elif len(cell_data.keys()) > 0:
        data, points = cell_data, mesh.cell_centers().points
    elif len(point_data.keys()) > 0:
        data, points = point_data, mesh.points
    else:
        raise ValueError("No point_data or cell_data arrays found in file.")

    M = None
    if vector_field_name is not None:
        if vector_field_name not in data:
            raise ValueError(f"'{vector_field_name}' not found. Available: {list(data.keys())}")
        M = np.asarray(data[vector_field_name])
        if M.ndim != 2 or M.shape[1] != 3:
            raise ValueError(f"'{vector_field_name}' is not a 3-component vector array.")
    else:
        vector_candidates = [k for k in data.keys()
                              if np.asarray(data[k]).ndim == 2 and np.asarray(data[k]).shape[1] == 3]
        preferred_names = ["M", "Magnetization", "magnetization", "moment", "spin", "Spin"]
        for name in preferred_names:
            if name in vector_candidates:
                M = np.asarray(data[name])
                break
        if M is None and len(vector_candidates) > 0:
            M = np.asarray(data[vector_candidates[0]])

        if M is None:
            component_sets = [("Mx", "My", "Mz"), ("mx", "my", "mz"),
                               ("M_x", "M_y", "M_z"), ("moment_x", "moment_y", "moment_z")]
            for cx, cy, cz in component_sets:
                if cx in data and cy in data and cz in data:
                    M = np.column_stack([np.asarray(data[cx]), np.asarray(data[cy]), np.asarray(data[cz])])
                    break

        if M is None:
            raise ValueError(
                "Could not auto-detect a magnetization vector field. "
                f"Available arrays: {list(data.keys())}. "
                "Pass vector_field_name explicitly."
            )

    return points, M, mesh

## 2. Average through thickness (project onto the x-y plane)

`average_through_thickness` bins every voxel/point into a regular 2D grid
in `(x, y)` (ignoring z), then averages all three magnetization components
over every sample that falls in each bin. Because it bins on `(x, y)`
rather than assuming a structured z-stack, it works whether your data
sits on an exact regular grid or an irregular/unstructured mesh:

- **Structured data:** if you don't pass `nx`/`ny`, it infers the grid
  resolution from the number of unique x and y coordinates, so you get
  back exactly the original in-plane grid, each cell averaged over z.
- **Unstructured data:** pass `nx`/`ny` yourself to choose the output
  resolution; points are binned (not interpolated), so it's robust to
  scattered/irregular samples.

Returns `x_centers`, `y_centers` (1D arrays) and `Mx_avg`, `My_avg`,
`Mz_avg` (2D arrays, shape `(len(y_centers), len(x_centers))`, i.e. ready
for `imshow`/`pcolormesh`). Empty bins are `NaN`.

In [ ]:
def average_through_thickness(points, M, nx=None, ny=None, thickness_axis=2):
    """
    Bin (x,y) and average M through the remaining axis (thickness_axis, default z).

    Parameters
    ----------
    points : (N,3) ndarray
    M : (N,3) ndarray
    nx, ny : int, optional
        Output grid resolution. If None, inferred from the number of
        unique coordinates along that axis (good for structured grids).
    thickness_axis : int
        Which column of `points` is the thickness direction to average
        over (0=x, 1=y, 2=z). Default 2 (z).

    Returns
    -------
    x_centers, y_centers : 1D ndarrays
    Mx_avg, My_avg, Mz_avg : 2D ndarrays, shape (ny, nx)
    """
    axes = [0, 1, 2]
    axes.remove(thickness_axis)
    ax_x, ax_y = axes

    x = points[:, ax_x]
    y = points[:, ax_y]

    x_unique = np.unique(np.round(x, decimals=8))
    y_unique = np.unique(np.round(y, decimals=8))

    if nx is None:
        nx = len(x_unique) if len(x_unique) > 1 else 100
    if ny is None:
        ny = len(y_unique) if len(y_unique) > 1 else 100

    x_edges = np.linspace(x.min(), x.max(), nx + 1)
    y_edges = np.linspace(y.min(), y.max(), ny + 1)
    x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
    y_centers = 0.5 * (y_edges[:-1] + y_edges[1:])

    count, _, _ = np.histogram2d(x, y, bins=[x_edges, y_edges])
    sums = []
    for comp in range(3):
        s, _, _ = np.histogram2d(x, y, bins=[x_edges, y_edges], weights=M[:, comp])
        sums.append(s)

    with np.errstate(invalid="ignore", divide="ignore"):
        avgs = [np.where(count > 0, s / count, np.nan) for s in sums]

    # transpose so array indexing is [row=y, col=x], matching imshow convention
    Mx_avg, My_avg, Mz_avg = [a.T for a in avgs]
    return x_centers, y_centers, Mx_avg, My_avg, Mz_avg

## 3. LTEM-style contrast plot

The standard way to visualize an in-plane magnetization/induction map
(the quantity LTEM is sensitive to) is a **color wheel** encoding:
hue = in-plane angle, brightness = in-plane magnitude (so regions where
the magnetization has tipped out of plane, e.g. a vortex core, show up
dark). A quiver overlay is optional.

`plot_ltem_contrast` builds this from the `Mx_avg`/`My_avg` arrays
returned above. `add_color_wheel_legend` adds a small color-wheel key to
the figure so the hue-to-angle mapping is unambiguous.

In [ ]:
def magnetization_to_rgb(Mx_avg, My_avg):
    """Map in-plane (Mx, My) to an RGB image: hue=angle, brightness=magnitude."""
    magnitude = np.sqrt(Mx_avg ** 2 + My_avg ** 2)
    angle = np.arctan2(My_avg, Mx_avg)
    hue = (angle + np.pi) / (2 * np.pi)
    max_mag = np.nanmax(magnitude)
    value = np.clip(magnitude / max_mag, 0, 1) if max_mag > 0 else np.zeros_like(magnitude)
    sat = np.ones_like(hue)
    hsv = np.nan_to_num(np.stack([hue, sat, value], axis=-1))
    return mcolors.hsv_to_rgb(hsv)


def plot_ltem_contrast(x_centers, y_centers, Mx_avg, My_avg, mode="color+quiver",
                        quiver_step=None, ax=None, figsize=(7, 6)):
    """
    Plot the thickness-averaged in-plane magnetization as an LTEM contrast proxy.

    mode : "color", "quiver", or "color+quiver"
    quiver_step : subsampling stride for the quiver arrows (auto if None)
    """
    rgb = magnetization_to_rgb(Mx_avg, My_avg)

    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    extent = [x_centers.min(), x_centers.max(), y_centers.min(), y_centers.max()]

    if "color" in mode:
        ax.imshow(rgb, origin="lower", extent=extent, aspect="equal")

    if "quiver" in mode:
        ny, nx = Mx_avg.shape
        if quiver_step is None:
            quiver_step = max(1, min(nx, ny) // 20)
        X, Y = np.meshgrid(x_centers, y_centers)
        sl = (slice(None, None, quiver_step), slice(None, None, quiver_step))
        arrow_color = "white" if "color" in mode else "black"
        ax.quiver(X[sl], Y[sl], Mx_avg[sl], My_avg[sl],
                   color=arrow_color, pivot="mid", width=0.003)
        if "color" not in mode:
            ax.set_aspect("equal")
            ax.set_xlim(extent[0], extent[1])
            ax.set_ylim(extent[2], extent[3])

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title("LTEM contrast proxy (thickness-averaged in-plane M)")
    return fig, ax


def add_color_wheel_legend(fig, rect=(0.78, 0.78, 0.15, 0.15)):
    """Add a small hue-wheel legend (angle -> color) to a figure."""
    wheel_ax = fig.add_axes(rect, projection="polar")
    n = 256
    theta = np.linspace(0, 2 * np.pi, n)
    r = np.linspace(0, 1, 2)
    T, R = np.meshgrid(theta, r)
    hue = T / (2 * np.pi)
    hsv = np.stack([hue, np.ones_like(hue), R], axis=-1)
    rgb = mcolors.hsv_to_rgb(hsv)
    wheel_ax.pcolormesh(T, R, np.zeros_like(T), color=rgb.reshape(-1, 3), shading="auto")
    wheel_ax.set_yticklabels([])
    wheel_ax.set_xticklabels([])
    wheel_ax.set_theta_zero_location("E")
    wheel_ax.spines["polar"].set_visible(False)
    wheel_ax.grid(False)
    return wheel_ax

## 4. Run it on your file

Replace `"your_file.vtk"` below with your actual path. If the vector field
isn't auto-detected, pass e.g. `vector_field_name="Magnetization"`.

In [ ]:
def plot_component_grayscale(x_centers, y_centers, comp_avg, label="$B_x$",
                              ax=None, cmap="gray", symmetric=True, figsize=(6, 6),
                              show_colorbar=True):
    """Plot a single averaged component (e.g. Mx_avg or My_avg) as a grayscale image."""
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    extent = [x_centers.min(), x_centers.max(), y_centers.min(), y_centers.max()]
    if symmetric:
        vmax = np.nanmax(np.abs(comp_avg))
        vmin = -vmax
    else:
        vmin, vmax = np.nanmin(comp_avg), np.nanmax(comp_avg)

    im = ax.imshow(comp_avg, origin="lower", extent=extent, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks([])
    ax.set_yticks([])

    if show_colorbar:
        cbar = fig.colorbar(im, ax=ax, orientation="horizontal", fraction=0.046, pad=0.04)
        cbar.set_label(label)
        cbar.set_ticks([vmin, 0, vmax])
        cbar.set_ticklabels(["-", "0", "+"])

    return fig, ax, im


def plot_bx_by_colorwheel_figure(x_centers, y_centers, Mx_avg, My_avg,
                                  raw_image=None, raw_label="electron counts",
                                  panel_labels=("a", "b", "c", "d"), figsize=None):
    """
    Assemble a multi-panel figure in the style of a typical LTEM data figure:
    optionally a raw/experimental image, then Bx, By grayscale panels, then
    the color-wheel combined map. Pass raw_image=None to skip that panel and
    just get (Bx, By, color-wheel).
    """
    panels = []
    if raw_image is not None:
        panels.append(("raw", raw_image))
    panels.append(("bx", Mx_avg))
    panels.append(("by", My_avg))
    panels.append(("wheel", None))

    n = len(panels)
    if figsize is None:
        figsize = (4 * n, 4.5)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]

    letter_iter = iter(panel_labels)
    extent = [x_centers.min(), x_centers.max(), y_centers.min(), y_centers.max()]

    for ax, (kind, arr) in zip(axes, panels):
        # letter = next(letter_iter, "")
        # ax.text(0.05, 0.95, f"({letter})", transform=ax.transAxes,
        #          ha="left", va="top", fontsize=12,
        #          bbox=dict(boxstyle="round", facecolor="white", edgecolor="black", pad=0.3))

        if kind == "raw":
            ax.imshow(arr, origin="lower", cmap="viridis")
            ax.set_xlabel(raw_label)
        elif kind == "bx":
            _, _, im = plot_component_grayscale(x_centers, y_centers, arr, label="Avg X component", ax=ax)
        elif kind == "by":
            _, _, im = plot_component_grayscale(x_centers, y_centers, arr, label="Avg Y component", ax=ax)
        elif kind == "wheel":
            rgb = magnetization_to_rgb(Mx_avg, My_avg)
            ax.imshow(rgb, origin="lower", extent=extent, aspect="equal")
            ax.set_xlabel(r"Predicted LTEM induction map")
            add_color_wheel_legend(fig, rect=[ax.get_position().x1 - 0.08,
                                               ax.get_position().y0 + 0.02, 0.07, 0.12])

        ax.set_xticks([])
        ax.set_yticks([])

    return fig, axes

In [ ]:
filename = r'C:\Users\emmab\Projects\Vector_Tomography\452_mag_const_interpolated_cropped.vtk'

points, M, mesh = read_vtk_magnetization(filename)
print(f"Loaded {points.shape[0]} points, magnetization array shape {M.shape}")
print("Available point_data fields:", list(mesh.point_data.keys()))
print("Available cell_data fields:", list(mesh.cell_data.keys()))

x_centers, y_centers, Mx_avg, My_avg, Mz_avg = average_through_thickness(points, M)


# Bx, By, and color-wheel map as a single figure (add raw_image=your_experimental_array
# and raw_label="electron counts" to also include a panel (a) like the reference figure)
fig2, axes2 = plot_bx_by_colorwheel_figure(x_centers, y_centers, Mx_avg, My_avg)
plt.show()


# fig, ax = plot_ltem_contrast(x_centers, y_centers, Mx_avg, My_avg, mode="color+quiver")
# add_color_wheel_legend(fig)
# plt.show()

## 5. Self-contained demo (synthetic magnetic vortex)

Since no sample file was available, this generates a synthetic 3D magnetic
vortex (in-plane curling magnetization with an out-of-plane core), saves
it as a legacy `.vtk` structured grid, and runs it through the exact same
pipeline above — so you can confirm everything works before pointing it
at your real data, and see what a known structure (vortex) should look
like in this representation: a full color-wheel rotation around the core,
with a dark spot at the core itself (magnetization tipped out of plane,
so its projected in-plane component vanishes there).

In [ ]:
def make_synthetic_vortex_vtk(filename="synthetic_vortex.vtk", nx=40, ny=40, nz=8, core_radius=0.2):
    """Generate a toy 3D magnetic vortex and save it as a legacy .vtk structured grid."""
    x = np.linspace(-1, 1, nx)
    y = np.linspace(-1, 1, ny)
    z = np.linspace(0, 1, nz)
    X, Y, Z = np.meshgrid(x, y, z, indexing="ij")

    r = np.sqrt(X ** 2 + Y ** 2)
    r_safe = np.where(r < 1e-6, 1e-6, r)

    Mz = np.exp(-(r / core_radius) ** 2)                 # out-of-plane core
    in_plane = np.sqrt(np.clip(1 - Mz ** 2, 0, 1))
    Mx = -Y / r_safe * in_plane                            # curling in-plane component
    My = X / r_safe * in_plane

    grid = pv.StructuredGrid(X, Y, Z)
    grid["M"] = np.column_stack([Mx.ravel(order="F"), My.ravel(order="F"), Mz.ravel(order="F")])
    grid.save(filename)
    return filename


demo_file = make_synthetic_vortex_vtk()

points, M, mesh = read_vtk_magnetization(demo_file)
x_centers, y_centers, Mx_avg, My_avg, Mz_avg = average_through_thickness(points, M)

fig, ax = plot_ltem_contrast(x_centers, y_centers, Mx_avg, My_avg, mode="color+quiver")
add_color_wheel_legend(fig)
plt.show()

## Notes / things to adjust for your data

- **Field name:** if `read_vtk_magnetization` can't auto-detect your
  vector field, check `mesh.point_data.keys()` (printed by the loader
  above) and pass `vector_field_name="..."` explicitly.
- **Thickness axis:** if your "thickness" direction isn't z, pass
  `thickness_axis=0` or `1` to `average_through_thickness`.
- **Resolution for unstructured data:** for scattered/irregular point
  clouds, set `nx`/`ny` explicitly in `average_through_thickness` to
  control the output grid resolution — too fine and you'll get lots of
  empty (NaN) bins, too coarse and you'll blur out real structure.
- **Units/normalization:** the color map normalizes brightness by the
  maximum in-plane magnitude in the averaged data
  (`value = |M_inplane| / max(|M_inplane|)`), so it's a relative map,
  not absolute induction in tesla.
- **Going physical:** this proxy shows you the projected in-plane
  induction, which is the direct input to a real LTEM phase/defocus
  simulation (e.g. Mansuripur algorithm or the transport-of-intensity
  equation). If you want that next step, `Mx_avg`/`My_avg` on this
  regular grid is exactly what such a calculation consumes — happy to
  build that on top of this if useful.

In [ ]:
import numpy as np

def check_grid_structure(points, thickness_axis=2, decimals=8):
    axes = [0, 1, 2]
    axes.remove(thickness_axis)
    ax_x, ax_y = axes

    x = np.round(points[:, ax_x], decimals)
    y = np.round(points[:, ax_y], decimals)
    z = np.round(points[:, thickness_axis], decimals)

    x_unique = np.unique(x)
    y_unique = np.unique(y)
    z_unique = np.unique(z)

    n_expected = len(x_unique) * len(y_unique) * len(z_unique)
    n_actual = len(points)

    print(f"Unique x values: {len(x_unique)}")
    print(f"Unique y values: {len(y_unique)}")
    print(f"Unique z (thickness) values: {len(z_unique)}")
    print(f"Expected points if fully structured grid: {n_expected}")
    print(f"Actual number of points: {n_actual}")

    if n_expected == n_actual:
        print("-> Point count matches a full structured grid.")
    else:
        print("-> Point count does NOT match a structured grid (could be irregular, missing points, or duplicates).")

    # Check how many z-layers each (x,y) column actually has
    xy_pairs = list(zip(x, y))
    from collections import Counter
    counts = Counter(xy_pairs)
    counts_values = np.array(list(counts.values()))

    print(f"\nNumber of unique (x,y) columns: {len(counts)}")
    print(f"Min points per column: {counts_values.min()}")
    print(f"Max points per column: {counts_values.max()}")
    print(f"Mean points per column: {counts_values.mean():.2f}")

    if counts_values.min() == counts_values.max():
        print(f"-> Every column has exactly {counts_values.min()} points — consistent with a structured grid.")
    else:
        print("-> Columns have varying point counts — this is NOT a clean structured grid.")

    return {
        "x_unique": x_unique,
        "y_unique": y_unique,
        "z_unique": z_unique,
        "counts_per_column": counts_values,
    }

In [ ]:
info = check_grid_structure(points, thickness_axis=2)

# tilting to see Neel character

In [ ]:
import numpy as np
from scipy.interpolate import RegularGridInterpolator
import matplotlib.pyplot as plt


def _grid_from_points(points, M, decimals=8):
    """
    Reshape (N,3) points/M into full 3D grid arrays, using coordinate
    values (not assumed ordering) to place each point in its slot.
    """
    x = np.round(points[:, 0], decimals)
    y = np.round(points[:, 1], decimals)
    z = np.round(points[:, 2], decimals)

    x_u, y_u, z_u = np.unique(x), np.unique(y), np.unique(z)
    ix = np.searchsorted(x_u, x)
    iy = np.searchsorted(y_u, y)
    iz = np.searchsorted(z_u, z)

    grid_shape = (len(x_u), len(y_u), len(z_u))
    Mgrid = np.zeros(grid_shape + (3,))
    filled = np.zeros(grid_shape, dtype=bool)
    Mgrid[ix, iy, iz, :] = M
    filled[ix, iy, iz] = True

    if not filled.all():
        print(f"Warning: {(~filled).sum()} of {filled.size} grid slots "
              f"are empty and being treated as vacuum (M=0).")

    return x_u, y_u, z_u, Mgrid


def build_M_interpolators(points, M, decimals=8):
    """
    Build trilinear interpolators for Mx, My, Mz. Outside the sample's
    bounding box, interpolation returns 0 (vacuum).
    """
    x_u, y_u, z_u, Mgrid = _grid_from_points(points, M, decimals=decimals)
    interpolators = [
        RegularGridInterpolator((x_u, y_u, z_u), Mgrid[..., c],
                                 method="linear", bounds_error=False,
                                 fill_value=0.0)
        for c in range(3)
    ]
    bounds = dict(xmin=x_u.min(), xmax=x_u.max(),
                  ymin=y_u.min(), ymax=y_u.max(),
                  zmin=z_u.min(), zmax=z_u.max())
    return interpolators, bounds


def _rotation_matrix(theta_deg, tilt_axis=0):
    """Rotation about x (tilt_axis=0) or y (tilt_axis=1). Beam (z) is fixed."""
    theta = np.deg2rad(theta_deg)
    c, s = np.cos(theta), np.sin(theta)
    R = np.eye(3)
    if tilt_axis == 0:      # tilt about x: mixes y,z
        R[1, 1], R[1, 2] = c, -s
        R[2, 1], R[2, 2] = s, c
    elif tilt_axis == 1:    # tilt about y: mixes x,z
        R[0, 0], R[0, 2] = c, s
        R[2, 0], R[2, 2] = -s, c
    else:
        raise ValueError("tilt_axis must be 0 (x) or 1 (y)")
    return R


def project_tilted_sample(points, M, theta_deg, tilt_axis=0,
                           nx=None, ny=None, n_z_samples=None,
                           normalize="path_length"):
    """
    Project Mx, My, Mz along the fixed beam (z) after rigidly tilting the
    sample by theta_deg about tilt_axis, rotating about the geometric
    center of the bounding box. Outside the original sample = vacuum.

    normalize : {"path_length", "original_thickness", "sum"}
    """
    interpolators, b = build_M_interpolators(points, M)
    R = _rotation_matrix(theta_deg, tilt_axis=tilt_axis)
    center = np.array([0.5*(b["xmin"]+b["xmax"]),
                        0.5*(b["ymin"]+b["ymax"]),
                        0.5*(b["zmin"]+b["zmax"])])

    x_u = np.unique(np.round(points[:, 0], 8))
    y_u = np.unique(np.round(points[:, 1], 8))
    nx = nx or (len(x_u) if len(x_u) > 1 else 100)
    ny = ny or (len(y_u) if len(y_u) > 1 else 100)

    x_edges = np.linspace(b["xmin"], b["xmax"], nx + 1)
    y_edges = np.linspace(b["ymin"], b["ymax"], ny + 1)
    x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
    y_centers = 0.5 * (y_edges[:-1] + y_edges[1:])

    thickness0 = b["zmax"] - b["zmin"]
    inplane_extent = max(b["xmax"] - b["xmin"], b["ymax"] - b["ymin"])
    th = np.deg2rad(abs(theta_deg))
    z_half = 1.2 * (0.5*thickness0*np.cos(th) + 0.5*inplane_extent*np.sin(th))
    z_min, z_max = center[2] - z_half, center[2] + z_half

    if n_z_samples is None:
        z_u = np.unique(np.round(points[:, 2], 8))
        n_layers = len(z_u) if len(z_u) > 1 else 8
        n_z_samples = max(4 * n_layers, 32)

    z_samples = np.linspace(z_min, z_max, n_z_samples)
    dz = z_samples[1] - z_samples[0]

    Xc, Yc = np.meshgrid(x_centers, y_centers, indexing="xy")  # (ny, nx)
    Msum = np.zeros((ny, nx, 3))
    path_length = np.zeros((ny, nx))
    R_inv_T = R  # since (R^T)^T = R, used below

    for z_k in z_samples:
        lab_pts = np.stack([Xc.ravel(), Yc.ravel(),
                             np.full(Xc.size, z_k)], axis=1)
        sample_pts = (lab_pts - center) @ R_inv_T + center  # inverse rotation

        M_sample = np.stack([f(sample_pts) for f in interpolators], axis=1)

        inside = (
            (sample_pts[:, 0] >= b["xmin"]) & (sample_pts[:, 0] <= b["xmax"]) &
            (sample_pts[:, 1] >= b["ymin"]) & (sample_pts[:, 1] <= b["ymax"]) &
            (sample_pts[:, 2] >= b["zmin"]) & (sample_pts[:, 2] <= b["zmax"])
        )

        M_lab = M_sample @ R.T  # rotate vectors into the lab frame

        Msum += (M_lab * dz).reshape(ny, nx, 3)
        path_length += (inside * dz).reshape(ny, nx)

    if normalize == "sum":
        Mx_avg, My_avg, Mz_avg = (Msum[..., c] for c in range(3))
    elif normalize == "original_thickness":
        with np.errstate(invalid="ignore", divide="ignore"):
            Mx_avg, My_avg, Mz_avg = (
                np.where(path_length > 0, Msum[..., c] / thickness0, np.nan)
                for c in range(3))
    elif normalize == "path_length":
        with np.errstate(invalid="ignore", divide="ignore"):
            Mx_avg, My_avg, Mz_avg = (
                np.where(path_length > 0, Msum[..., c] / path_length, np.nan)
                for c in range(3))
    else:
        raise ValueError("normalize must be 'path_length', 'original_thickness', or 'sum'")

    return x_centers, y_centers, Mx_avg, My_avg, Mz_avg


def tilt_series(points, M, angles_deg, tilt_axis=0, normalize="path_length", **kwargs):
    """angles_deg: e.g. np.arange(-20, 21, 5). Returns {angle: (x_c, y_c, Mx, My, Mz)}."""
    return {theta: project_tilted_sample(points, M, theta, tilt_axis=tilt_axis,
                                          normalize=normalize, **kwargs)
            for theta in angles_deg}


def plot_tilt_series(results, component="Mx", cmap="RdBu_r", vmin=None, vmax=None):
    comp_idx = {"Mx": 2, "My": 3, "Mz": 4}[component]
    angles = sorted(results.keys())
    ncols = min(len(angles), 5)
    nrows = int(np.ceil(len(angles) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3*nrows), squeeze=False)

    if vmin is None or vmax is None:
        vals = np.concatenate([results[a][comp_idx][~np.isnan(results[a][comp_idx])]
                                for a in angles])
        vmax_auto = np.nanmax(np.abs(vals)) if vals.size else 1.0
        vmin = -vmax_auto if vmin is None else vmin
        vmax = vmax_auto if vmax is None else vmax

    for i, theta in enumerate(angles):
        ax = axes[i // ncols][i % ncols]
        x_c, y_c, Mx, My, Mz = results[theta]
        data = {"Mx": Mx, "My": My, "Mz": Mz}[component]
        im = ax.imshow(data, origin="lower",
                        extent=[x_c.min(), x_c.max(), y_c.min(), y_c.max()],
                        cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(f"{theta:g}°"); ax.set_xticks([]); ax.set_yticks([])

    for j in range(len(angles), nrows*ncols):
        axes[j // ncols][j % ncols].axis("off")

    fig.colorbar(im, ax=axes, shrink=0.6, label=component)
    fig.suptitle(f"{component} vs. tilt angle")
    return fig, axes

In [ ]:
results = tilt_series(points, M, angles_deg=np.arange(-20, 21, 5), tilt_axis=0)
plot_tilt_series(results, component="Mx")
plot_tilt_series(results, component="My")

In [ ]:
results = tilt_series(points, M, angles_deg=np.arange(-20, 21, 5), tilt_axis=0, normalize="original_thickness")
plot_tilt_series(results, component="Mx")
plot_tilt_series(results, component="My")